In [1]:
# Importação de bibliotecas, configuração de caminhos e módulos do projeto (src)

import sys
import os

import torch

# Adiciona o diretório pai (raiz do projeto) ao caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))

from src.config import (
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
    NUMERIC_FEATURES,
    REGRESSION_CATEGORICAL_FEATURES,
    CLASSIFICATION_CATEGORICAL_FEATURES,
    CLASS_NAMES,
)

from src.data.load_data import (
    load_regression_data,
    load_classification_data,
)

from src.data.split_data import (
    split_regression_data,
    split_classification_data,
)

from src.data.preprocessing import (
    preprocess_features,
    scale_regression_target,
    encode_classification_target,
)

from src.utils.reproducibility import set_seed

from src.data.torch_data import (
    create_tensor_datasets,
    create_dataloaders,
)

from pathlib import Path
import joblib

In [2]:
# Carregamento dos dados brutos de regressão e classificação e conferência das dimensões

df_regression = load_regression_data()
df_classification = load_classification_data()

print("REGRESSÃO:", df_regression.shape)
print("CLASSIFICAÇÃO:", df_classification.shape)

REGRESSÃO: (50000, 9)
CLASSIFICAÇÃO: (50000, 10)


In [3]:
# Divisão dos conjuntos em treino, validação e teste (com estratificação para classificação)

regression_splits = split_regression_data(
    df_regression,
    REGRESSION_TARGET,
)

classification_splits = split_classification_data(
    df_classification,
    CLASSIFICATION_TARGET,
)

In [4]:
# Exibição da quantidade de registros em cada split dos conjuntos de dados

def show_split_sizes(splits, title):
    print(f"\n{title}")

    for name, values in splits.items():
        print(f"{name}: {len(values)} registros")


show_split_sizes(regression_splits, "REGRESSÃO")
show_split_sizes(classification_splits, "CLASSIFICAÇÃO")


REGRESSÃO
X_train: 32000 registros
X_val: 8000 registros
X_test: 10000 registros
y_train: 32000 registros
y_val: 8000 registros
y_test: 10000 registros

CLASSIFICAÇÃO
X_train: 32000 registros
X_val: 8000 registros
X_test: 10000 registros
y_train: 32000 registros
y_val: 8000 registros
y_test: 10000 registros


In [5]:
# Verificação da distribuição e proporção das classes nos conjuntos de treino, validação e teste

for split_name in ["train", "val", "test"]:
    y = classification_splits[f"y_{split_name}"]

    print(f"\n{split_name.upper()}")
    print(y.value_counts())
    print((y.value_counts(normalize=True) * 100).round(2))


TRAIN
Diabetes_Risk
High        23419
Moderate     8280
Low           301
Name: count, dtype: int64
Diabetes_Risk
High        73.18
Moderate    25.87
Low          0.94
Name: proportion, dtype: float64

VAL
Diabetes_Risk
High        5855
Moderate    2070
Low           75
Name: count, dtype: int64
Diabetes_Risk
High        73.19
Moderate    25.87
Low          0.94
Name: proportion, dtype: float64

TEST
Diabetes_Risk
High        7319
Moderate    2587
Low           94
Name: count, dtype: int64
Diabetes_Risk
High        73.19
Moderate    25.87
Low          0.94
Name: proportion, dtype: float64


In [6]:
# Pré-processamento das variáveis preditoras (padronização numérica e codificação categórica)

regression_features = preprocess_features(
    regression_splits,
    NUMERIC_FEATURES,
    REGRESSION_CATEGORICAL_FEATURES,
)

classification_features = preprocess_features(
    classification_splits,
    NUMERIC_FEATURES,
    CLASSIFICATION_CATEGORICAL_FEATURES,
)

In [7]:
# Padronização do alvo de regressão (StandardScaler) e codificação dos rótulos de classificação

# Regressão: padronização do alvo
regression_targets = scale_regression_target(
    regression_splits
)

# Classificação: codificação dos rótulos
classification_targets = encode_classification_target(
    classification_splits
)

In [8]:
# Criação do diretório de dados processados e persistência do scaler do alvo de regressão

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    regression_targets["target_scaler"],
    PROCESSED_DIR / "regression_target_scaler.pkl"
)

['..\\data\\processed\\regression_target_scaler.pkl']

In [9]:
# Inspeção dimensional dos dados pré-processados e lista de features finais

print("REGRESSÃO")
print("Features:", regression_features["X_train"].shape)
print("Alvo:", regression_targets["y_train"].shape)
print("Quantidade de features:",
      len(regression_features["feature_names"]))

print("\nCLASSIFICAÇÃO")
print("Features:", classification_features["X_train"].shape)
print("Alvo:", classification_targets["y_train"].shape)
print("Quantidade de features:",
      len(classification_features["feature_names"]))

print("\nClasses:", CLASS_NAMES)
print("\nFeatures da regressão:")
print(regression_features["feature_names"])

print("\nFeatures da classificação:")
print(classification_features["feature_names"])

REGRESSÃO
Features: (32000, 15)
Alvo: (32000, 1)
Quantidade de features: 15

CLASSIFICAÇÃO
Features: (32000, 15)
Alvo: (32000,)
Quantidade de features: 15

Classes: ['Low', 'Moderate', 'High']

Features da regressão:
['Fasting_Blood_Sugar', 'HbA1c', 'Age', 'BMI', 'Family_History_Diabetes_No', 'Family_History_Diabetes_Yes', 'Hypertension_No', 'Hypertension_Yes', 'Diet_Quality_Average', 'Diet_Quality_Healthy', 'Diet_Quality_Poor', 'Physical_Activity_Level_High', 'Physical_Activity_Level_Low', 'Physical_Activity_Level_Moderate', 'Physical_Activity_Level_Não_informado']

Features da classificação:
['Fasting_Blood_Sugar', 'HbA1c', 'Age', 'BMI', 'Family_History_Diabetes_No', 'Family_History_Diabetes_Yes', 'Hypertension_No', 'Hypertension_Yes', 'Diet_Quality_Average', 'Diet_Quality_Healthy', 'Diet_Quality_Poor', 'PCOS_No', 'PCOS_Yes', 'Heart_Disease_No', 'Heart_Disease_Yes']


In [10]:
# Conversão dos dados de regressão em TensorDataset e instanciação dos DataLoaders

regression_datasets = create_tensor_datasets(
    features=regression_features,
    targets=regression_targets,
    task="regression",
)

regression_loaders = create_dataloaders(
    datasets=regression_datasets,
    batch_size=64,
    seed=42,
)

In [11]:
# Conversão dos dados de classificação em TensorDataset e instanciação dos DataLoaders

classification_datasets = create_tensor_datasets(
    features=classification_features,
    targets=classification_targets,
    task="classification",
)

classification_loaders = create_dataloaders(
    datasets=classification_datasets,
    batch_size=64,
    seed=42,
)

In [12]:
# Validação e inspeção do primeiro lote (batch) de tensores para regressão e classificação
X_reg, y_reg = next(iter(regression_loaders["train"]))

print("REGRESSÃO")
print("Features:", X_reg.shape)
print("Alvo:", y_reg.shape)
print("Tipo das features:", X_reg.dtype)
print("Tipo do alvo:", y_reg.dtype)

print("\nCLASSIFICAÇÃO")

# Primeiro lote de classificação
X_clf, y_clf = next(iter(classification_loaders["train"]))

print("Features:", X_clf.shape)
print("Alvo:", y_clf.shape)
print("Tipo das features:", X_clf.dtype)
print("Tipo do alvo:", y_clf.dtype)

print("\nClasses presentes no lote:", torch.unique(y_clf))

REGRESSÃO
Features: torch.Size([64, 15])
Alvo: torch.Size([64, 1])
Tipo das features: torch.float32
Tipo do alvo: torch.float32

CLASSIFICAÇÃO
Features: torch.Size([64, 15])
Alvo: torch.Size([64])
Tipo das features: torch.float32
Tipo do alvo: torch.int64

Classes presentes no lote: tensor([0, 1, 2])


In [13]:
# Conferência da quantidade total de lotes por conjunto (treino, validação e teste)

print("REGRESSÃO")
for split, loader in regression_loaders.items():
    print(f"{split}: {len(loader)} lotes")

print("\nCLASSIFICAÇÃO")
for split, loader in classification_loaders.items():
    print(f"{split}: {len(loader)} lotes")

REGRESSÃO
train: 500 lotes
val: 125 lotes
test: 157 lotes

CLASSIFICAÇÃO
train: 500 lotes
val: 125 lotes
test: 157 lotes


In [14]:
# Salvamento dos datasets de tensores PyTorch (.pt) no diretório de dados processados
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Salvar os datasets
torch.save(
    regression_datasets,
    processed_dir / "regression_data.pt"
)

torch.save(
    classification_datasets,
    processed_dir / "classification_data.pt"
)

In [15]:
# Persistência dos pré-processadores e scalers via Joblib (.pkl)

joblib.dump(
    regression_features["preprocessor"],
    processed_dir / "regression_preprocessor.pkl"
)

joblib.dump(
    classification_features["preprocessor"],
    processed_dir / "classification_preprocessor.pkl"
)

joblib.dump(
    regression_targets["target_scaler"],
    processed_dir / "regression_target_scaler.pkl"
)

['..\\data\\processed\\regression_target_scaler.pkl']

In [16]:
# Salvamento das listas oficiais com os nomes das features pré-processadas (.pt)

torch.save(
    regression_features["feature_names"],
    processed_dir / "regression_feature_names.pt"
)

torch.save(
    classification_features["feature_names"],
    processed_dir / "classification_feature_names.pt"
)

In [17]:
# Seleção e persistência de 5 amostras reais de teste para validação futura de inferência
regression_samples = regression_splits["X_test"].iloc[:5].copy()

classification_samples = classification_splits["X_test"].iloc[:5].copy()

# Salvar as amostras
torch.save(
    regression_samples,
    processed_dir / "regression_samples.pt"
)

torch.save(
    classification_samples,
    processed_dir / "classification_samples.pt"
)